CNN

1: Importación de Librerías y Configuración del Entorno

In [1]:
import os
import re
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Configuración de rutas
BASE_DIR = './x-rays'
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
VAL_DIR = os.path.join(BASE_DIR, 'validation')
TEST_DIR = os.path.join(BASE_DIR, 'test')
LEADERBOARD_DIR = os.path.join(BASE_DIR, 'leaderboard')
SUBMISSION_PATH = os.path.join(BASE_DIR, 'submission_levi.csv')

IMG_SIZE = (150, 150)
BATCH_SIZE = 32
EPOCHS = 35

def crear_df_desde_directorio(directorio):
    rutas, etiquetas = [], []
    for clase in os.listdir(directorio):
        ruta_clase = os.path.join(directorio, clase)
        if os.path.isdir(ruta_clase):
            for archivo in os.listdir(ruta_clase):
                if archivo.lower().endswith(('.png', '.jpg', '.jpeg')):
                    rutas.append(os.path.join(ruta_clase, archivo))
                    etiquetas.append(clase)
    return pd.DataFrame({'filename': rutas, 'label': etiquetas})

df_train = crear_df_desde_directorio(TRAIN_DIR)
df_val = crear_df_desde_directorio(VAL_DIR)
df_test = crear_df_desde_directorio(TEST_DIR)

print(f"Imágenes de entrenamiento: {len(df_train)}")
print(f"Imágenes de validación: {len(df_val)}")

Imágenes de entrenamiento: 351
Imágenes de validación: 90


2: Definición de Rutas e Hiperparámetros Globales

In [2]:
# MAPEO EXPLÍCITO DE CLASES (Asegúrate de que coincida con las reglas del challenge)
class_mapping = {
    'COVID-19': 0,
    'HEALTHY': 1,
    'PNEUMONIA': 2
}

# Convertimos las etiquetas de texto a strings numéricos para que ImageDataGenerator los procese correctamente
df_train['label'] = df_train['label'].map(class_mapping).astype(str)
df_val['label'] = df_val['label'].map(class_mapping).astype(str)
df_test['label'] = df_test['label'].map(class_mapping).astype(str)

# Generadores de datos con Normalización [0, 1] incluida
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_dataframe(
    df_train, x_col='filename', y_col='label',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=True, seed=42
)

val_generator = val_test_datagen.flow_from_dataframe(
    df_val, x_col='filename', y_col='label',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

test_generator = val_test_datagen.flow_from_dataframe(
    df_test, x_col='filename', y_col='label',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

print(f"Mapeo interno final del generador: {train_generator.class_indices}")

Found 351 validated image filenames belonging to 3 classes.
Found 90 validated image filenames belonging to 3 classes.
Found 96 validated image filenames belonging to 3 classes.
Mapeo interno final del generador: {'0': 0, '1': 1, '2': 2}


3: Carga de Datos y Canalización con Data Augmentation

In [3]:
model = Sequential([
    Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.2),
    
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),
    
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.3),
    
    Flatten(),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.4),
    Dense(3, activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

4: Arquitectura del Modelo CNN Regularizado

In [4]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6, verbose=1)
]

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks
)

Epoch 1/35
11/11 ━━━━━━━━━━━━━━━━━━━━ 33s 3s/step - accuracy: 0.7379 - loss: 0.9382 - val_accuracy: 0.3444 - val_loss: 1.7008 - learning_rate: 0.0010
Epoch 2/35
11/11 ━━━━━━━━━━━━━━━━━━━━ 28s 3s/step - accuracy: 0.8291 - loss: 0.4916 - val_accuracy: 0.3333 - val_loss: 4.2652 - learning_rate: 0.0010
Epoch 3/35
11/11 ━━━━━━━━━━━━━━━━━━━━ 30s 3s/step - accuracy: 0.8120 - loss: 0.5058 - val_accuracy: 0.3778 - val_loss: 8.5651 - learning_rate: 0.0010
Epoch 4/35
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8348 - loss: 0.4488
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.
11/11 ━━━━━━━━━━━━━━━━━━━━ 35s 2s/step - accuracy: 0.8348 - loss: 0.4488 - val_accuracy: 0.3778 - val_loss: 15.2912 - learning_rate: 0.0010
Epoch 5/35
11/11 ━━━━━━━━━━━━━━━━━━━━ 21s 2s/step - accuracy: 0.8547 - loss: 0.4077 - val_accuracy: 0.3444 - val_loss: 22.3586 - learning_rate: 2.0000e-04
Epoch 6/35
11/11 ━━━━━━━━━━━━━━━━━━━━ 24s 2s/step - accuracy: 0.8632 - loss: 0.3716 - val_accur

5: Entrenamiento con Callbacks Inteligentes

In [5]:
def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]

def generar_entrega_definitiva(modelo, ruta_leaderboard, ruta_salida):
    archivos = [f for f in os.listdir(ruta_leaderboard) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    archivos.sort(key=natural_sort_key)
    
    ids, predicciones = [], []
    
    print(f"Procesando {len(archivos)} imágenes para la entrega...")
    for idx, nombre_archivo in enumerate(archivos):
        id_numerico = int(os.path.splitext(nombre_archivo)[0])
        ruta_img = os.path.join(ruta_leaderboard, nombre_archivo)
        
        # Preprocesado idéntico al generador
        img = load_img(ruta_img, target_size=IMG_SIZE)
        img_array = img_to_array(img) / 255.0
        img_array = np.expand_dims(img_array, axis=0)
        
        preds = modelo.predict(img_array, verbose=0)
        clase_predicha = np.argmax(preds, axis=1)[0]
        
        ids.append(id_numerico)
        predicciones.append(clase_predicha)
        
        if idx < 3 or idx >= len(archivos) - 3:
            print(f"Imagen: {nombre_archivo} -> ID: {id_numerico} -> Clase Predicha: {clase_predicha}")

    df_sub = pd.DataFrame({'id': ids, 'prediction': predicciones})
    df_sub.to_csv(ruta_salida, index=False)
    print(f"\nArchivo exportado exitosamente en: {ruta_salida}")
    return df_sub

df_final = generar_entrega_definitiva(model, LEADERBOARD_DIR, SUBMISSION_PATH)
print(df_final.head(10))

Procesando 116 imágenes para la entrega...
Imagen: 1.png -> ID: 1 -> Clase Predicha: 0
Imagen: 2.png -> ID: 2 -> Clase Predicha: 2
Imagen: 3.png -> ID: 3 -> Clase Predicha: 2
Imagen: 114.png -> ID: 114 -> Clase Predicha: 2
Imagen: 115.png -> ID: 115 -> Clase Predicha: 2
Imagen: 116.png -> ID: 116 -> Clase Predicha: 0

Archivo exportado exitosamente en: ./x-rays\submission_levi.csv
   id  prediction
0   1           0
1   2           2
2   3           2
3   4           2
4   5           0
5   6           2
6   7           2
7   8           2
8   9           2
9  10           2


6: Evaluación de Métricas en el Conjunto de Test

7: Generación Corrección de Inferencia y Generación de Entrega (Leaderboard)